# Fase 9 — Análisis en profundidad de Golazo

Continúa el EDA de la Fase 8 con preguntas de negocio concretas: qué caracteriza a los días fuertes, cómo rinde la categoría 'Afición', si los vídeos virales coinciden con los picos de audiencia, cuáles son los vídeos más completos del canal, qué se publica cuando más se gana en suscriptores, y si el calendario de publicación aprovecha el pico real de audiencia (sábado).

**Requisito previo:** base de datos `golazo_growup` ya cargada (`python -m src.cargar_datos`).

## 0. Advertencia sobre granularidad de los datos

Antes de interpretar nada, importante tener claro qué nivel de detalle tiene cada tabla — condiciona cómo leer varios de los puntos siguientes:

- `evolucion_diaria`, `trafico`, `demografia`, `ingresos` → agregados a **nivel de canal**, por periodo completo (no por vídeo, no por categoría). No se puede saber, por ejemplo, cuánto tráfico de 'sugeridos' generó un vídeo concreto, o cuánto ingresó la categoría 'Afición' en particular.
- `retencion_audiencia` → solo existe para una **muestra de 15 vídeos** (de 338), no el catálogo completo.
- `video.views_totales` → es el acumulado de **toda la vida** del vídeo, no las vistas que recibió específicamente el día de su publicación. Cruzar 'día de publicación' con 'día de más vistas del canal' no es estrictamente la misma variable — se hace en este notebook porque es la aproximación disponible con los datos actuales, pero se señala en cada punto donde aplica.

## 0.1 Carga de datos y columnas derivadas

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

from src import db_connection as dbc

url = dbc.DATABASE_URL or (
    f"postgresql+psycopg2://{dbc.DB_CONFIG['user']}:{dbc.DB_CONFIG['password']}"
    f"@{dbc.DB_CONFIG['host']}:{dbc.DB_CONFIG['port']}/{dbc.DB_CONFIG['dbname']}"
)
if url.startswith('postgresql://'):
    url = url.replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(url)

canal = pd.read_sql('SELECT * FROM canal', engine)
video = pd.read_sql('SELECT * FROM video', engine, parse_dates=['fecha_publicacion'])
video_tag = pd.read_sql('SELECT * FROM video_tag', engine)
evolucion = pd.read_sql('SELECT * FROM evolucion_diaria', engine, parse_dates=['fecha'])
retencion = pd.read_sql('SELECT * FROM retencion_audiencia', engine)
trafico = pd.read_sql('SELECT * FROM trafico', engine)
ingresos = pd.read_sql('SELECT * FROM ingresos', engine, parse_dates=['fecha'])

# Columnas derivadas (mismas que en la Fase 8, recalculadas aquí para que este notebook sea autónomo)
evolucion['suscriptores_netos'] = evolucion['subscribers_gained'] - evolucion['subscribers_lost']
evolucion['dia_semana'] = evolucion['fecha'].dt.day_name()
evolucion['es_finde'] = evolucion['dia_semana'].isin(['Saturday', 'Sunday'])

video['ratio_likes_vista'] = video['likes'] / video['views_totales']
video['ratio_comentarios_vista'] = video['comentarios'] / video['views_totales']
video['z_views_categoria'] = video.groupby('categoria')['views_totales'].transform(
    lambda s: (s - s.mean()) / s.std()
)
video['dia_semana_publicacion'] = video['fecha_publicacion'].dt.day_name()
fecha_corte = video['fecha_publicacion'].max() - pd.Timedelta(days=30)
video['es_reciente'] = video['fecha_publicacion'] > fecha_corte

retencion_media_por_video = retencion.groupby('video_id')['audience_watch_ratio'].mean().rename('retencion_media')
video = video.merge(retencion_media_por_video, on='video_id', how='left')

print(f'Vídeos: {len(video)} | Con retención disponible: {video["retencion_media"].notna().sum()}')

## 1. Los dos días de mayor audiencia media: qué los caracteriza

Objetivo: identificar qué se publicó, qué pasó con las suscripciones y el engagement esos días, para poder **replicar el patrón** en el futuro.

In [ ]:
top2_dias_audiencia = evolucion.sort_values('views', ascending=False).head(2)
print('Los 2 días de mayor audiencia media (por vistas):')
display(top2_dias_audiencia[['fecha', 'dia_semana', 'views', 'subscribers_gained', 'subscribers_lost', 'suscriptores_netos']])

media_views = evolucion['views'].mean()
media_subs = evolucion['subscribers_gained'].mean()
for _, fila in top2_dias_audiencia.iterrows():
    pct_views = (fila['views'] / media_views - 1) * 100
    pct_subs = (fila['subscribers_gained'] / media_subs - 1) * 100
    print(f"{fila['fecha'].date()} ({fila['dia_semana']}): {pct_views:+.0f}% vistas y {pct_subs:+.0f}% "
          f"suscriptores ganados respecto a la media del periodo.")

In [ ]:
# Vídeos publicados esos días concretos (aproximación: fecha de publicación == día pico)
videos_dias_top = video[video['fecha_publicacion'].isin(top2_dias_audiencia['fecha'])]

print(f'Vídeos publicados justo esos 2 días: {len(videos_dias_top)}')
if len(videos_dias_top):
    display(videos_dias_top[['titulo', 'categoria', 'fecha_publicacion', 'views_totales', 'likes', 'comentarios']])
    print('\nCategoría(s) implicada(s):', videos_dias_top['categoria'].unique().tolist())
else:
    print('Ningún vídeo se publicó exactamente esos días — el pico de audiencia esos días probablemente '
          'proviene de vídeos publicados antes que siguen acumulando vistas, no de una subida nueva. '
          'Esto es esperable: views_totales es acumulado de vida, no vistas del día.')

**Para reforzar este punto profesionalmente:** con la granularidad actual no se puede aislar qué vídeo concreto generó las vistas de ese día (`views_totales` no está fechado día a día por vídeo). Para una versión de producción real, el dato que de verdad hace falta aquí es la Analytics API **a nivel de vídeo** (`filters=video==ID`, dimensión `day`), que permitiría saber exactamente qué vídeo tira del pico cada día — algo que con los datos actuales solo podemos aproximar. Aun así, el patrón de **día de la semana** (¿caen en fin de semana?) y la relación con `subscribers_gained` ese mismo día sí son datos reales y accionables: si ambos días son sábado/domingo, refuerza la idea de coordinar las publicaciones más fuertes con la jornada de liga.

## 2. Categoría 'Afición' en profundidad

Qué la caracteriza frente al resto del canal, y qué palancas reales de datos existen para mejorar su retención y difusión — análisis sobre los vídeos completos de la categoría, sin muestreo.

In [ ]:
aficion = video[video['categoria'] == 'Afición']
print(f'Vídeos de la categoría Afición: {len(aficion)} de {len(video)} totales')
display(aficion[['duracion_segundos', 'views_totales', 'likes', 'comentarios',
                  'ratio_likes_vista', 'ratio_comentarios_vista', 'retencion_media']].describe().round(3))

In [ ]:
# Afición vs. resto de categorías, en TODAS las métricas relevantes disponibles a nivel de vídeo
comparativa_aficion = video.groupby(video['categoria'] == 'Afición')[
    ['views_totales', 'likes', 'comentarios', 'ratio_likes_vista', 'ratio_comentarios_vista',
     'duracion_segundos', 'retencion_media']
].mean()
comparativa_aficion.index = comparativa_aficion.index.map({True: 'Afición', False: 'Resto de categorías'})
print('Afición vs. resto de categorías (medias, 338 vídeos completos):')
display(comparativa_aficion.round(3))

diferencia_retencion = (
    comparativa_aficion.loc['Afición', 'retencion_media'] - comparativa_aficion.loc['Resto de categorías', 'retencion_media']
) * 100
print(f"\nAfición retiene {diferencia_retencion:+.1f} puntos porcentuales frente al resto del canal.")

In [ ]:
# Tags más frecuentes en Afición, cruzados con el rendimiento de los vídeos que los llevan
tags_aficion = video_tag[video_tag['video_id'].isin(aficion['video_id'])]
top_tags_aficion = tags_aficion['tag'].value_counts().head(5)
print('Tags más usados en Afición:')
display(top_tags_aficion)

for tag in top_tags_aficion.index:
    videos_con_tag = aficion[aficion['video_id'].isin(tags_aficion[tags_aficion['tag'] == tag]['video_id'])]
    print(f"  '{tag}': retención media {videos_con_tag['retencion_media'].mean()*100:.1f}%, "
          f"engagement (likes/vista) {videos_con_tag['ratio_likes_vista'].mean()*100:.2f}%")

In [ ]:
# Duración vs. retención, específicamente DENTRO de Afición (no a nivel de canal general)
corr_duracion_retencion_aficion = aficion[['duracion_segundos', 'retencion_media']].corr().iloc[0, 1]
print(f'Correlación duración-retención SOLO dentro de Afición: {corr_duracion_retencion_aficion:.3f}')

franja_aficion = aficion.groupby(aficion['duracion_segundos'].apply(
    lambda s: 'Corto (<5 min)' if s/60 < 5 else ('Medio (5-15 min)' if s/60 <= 15 else 'Largo (>15 min)')
))[['views_totales', 'retencion_media', 'ratio_likes_vista']].mean()
print('\nRendimiento de Afición por franja de duración:')
display(franja_aficion.round(3))

**Nota de granularidad:** se ha dejado fuera de este punto la relación con ingresos y fuentes de tráfico porque en este proyecto ambos solo existen agregados a nivel de canal/día (Fase 3), nunca por categoría — incluirlos aquí habría significado repartir una cifra de canal entre categorías de forma arbitraria, no medirla. Todo lo anterior, en cambio, es dato real a nivel de vídeo para los 338 vídeos, incluida ya la retención completa.

## 3. ¿Los vídeos con vistas anormalmente altas coinciden con los días de mayor audiencia?

In [ ]:
picos_virales = video[video['z_views_categoria'] > 2]
print(f'Vídeos con vistas anormalmente altas para su categoría: {len(picos_virales)} de {len(video)}')

coinciden = picos_virales[picos_virales['fecha_publicacion'].isin(top2_dias_audiencia['fecha'])]
print(f'De esos, publicados justo en los 2 días de mayor audiencia: {len(coinciden)}')

if len(coinciden) == 0:
    print('\nNo coinciden por fecha exacta de publicación. Esto es esperable: views_totales acumula vistas '
          'de toda la vida del vídeo, así que un vídeo viral pudo publicarse semanas antes del día de pico '
          'de audiencia del canal, y seguir sumando vistas ese día concreto sin haberse publicado entonces.')
else:
    display(coinciden[['titulo', 'categoria', 'fecha_publicacion', 'views_totales']])

In [ ]:
# Ya que no coinciden por fecha, comparamos el perfil COMPLETO de los vídeos virales frente al resto
# (los 338 vídeos, incluida ya la retención completa — antes esto no se podía hacer con solo 15 vídeos)
resto = video[video['z_views_categoria'] <= 2]

perfil_comparado = pd.DataFrame({
    'Vídeos virales': [
        picos_virales['duracion_segundos'].mean() / 60,
        picos_virales['ratio_likes_vista'].mean() * 100,
        picos_virales['ratio_comentarios_vista'].mean() * 100,
        picos_virales['retencion_media'].mean() * 100,
    ],
    'Resto de vídeos': [
        resto['duracion_segundos'].mean() / 60,
        resto['ratio_likes_vista'].mean() * 100,
        resto['ratio_comentarios_vista'].mean() * 100,
        resto['retencion_media'].mean() * 100,
    ],
}, index=['Duración media (min)', 'Likes/vista (%)', 'Comentarios/vista (%)', 'Retención media (%)'])

print('Perfil comparado: vídeos virales vs. resto (338 vídeos completos):')
display(perfil_comparado.round(2))

print('\nCategoría de los vídeos virales:')
display(picos_virales['categoria'].value_counts())
print('\nDía de la semana de publicación de los vídeos virales:')
display(picos_virales['dia_semana_publicacion'].value_counts())

**Lectura profesional:** si los vídeos virales también retienen mejor (no solo tienen más vistas), es una señal de que su éxito es de calidad de contenido, replicable — si retienen igual o peor que la media pese a tener muchas más vistas, es más probable que el pico venga de un factor externo puntual (viralidad de alcance, no de enganche), y conviene no sobre-invertir en imitar la fórmula sin más evidencia.

## 4. Top 5 de vídeos "perfectos"

Ranking único sobre los **338 vídeos**, combinando alcance, engagement y retención — ya no hace falta partir en dos rankings porque la retención ahora cubre el catálogo completo, no una muestra de 15.

In [ ]:
def normalizar(serie):
    return (serie - serie.min()) / (serie.max() - serie.min())

video['score_perfecto'] = (
    normalizar(video['views_totales']) * 0.35
    + normalizar(video['likes']) * 0.25
    + normalizar(video['comentarios']) * 0.15
    + normalizar(video['retencion_media']) * 0.25
)

top5_perfectos = video.sort_values('score_perfecto', ascending=False).head(5)
print('TOP 5 vídeos "perfectos" (338 vídeos, score único: alcance + engagement + retención):')
display(top5_perfectos[['titulo', 'categoria', 'views_totales', 'likes', 'comentarios',
                         'retencion_media', 'score_perfecto']].round(3))

**Por qué no incluye monetización por vídeo:** en este proyecto, los ingresos (Fase 3) solo se definieron con dimensión `day` a nivel de canal, nunca con dimensión `video` — la YouTube Analytics API real sí permite pedir ingresos con dimensión `video`, pero no se contempló en el diseño original de este proyecto. Añadir una cifra de "euros por vídeo" aquí sería inventar un dato, no derivarlo. Si en una futura fase se quiere monetización real por vídeo, el cambio se haría en `analytics_client.py` (Fase 3), añadiendo `dimensions="video"` al informe de ingresos, y regenerando los datos sintéticos en consecuencia.

## 5. Qué se publica los días de mayor ganancia de suscriptores (+ el día siguiente)

In [ ]:
top3_dias_subs = evolucion.sort_values('subscribers_gained', ascending=False).head(3)
print('Los 3 días con más suscriptores ganados:')
display(top3_dias_subs[['fecha', 'dia_semana', 'subscribers_gained', 'views']])

In [ ]:
for _, fila in top3_dias_subs.iterrows():
    fecha = fila['fecha']
    videos_ese_dia = video[video['fecha_publicacion'] == fecha]
    categorias = videos_ese_dia['categoria'].tolist() if len(videos_ese_dia) else ['(ninguno publicado ese día)']

    dia_siguiente = evolucion[evolucion['fecha'] == fecha + pd.Timedelta(days=1)]
    subs_siguiente = dia_siguiente['subscribers_gained'].iloc[0] if len(dia_siguiente) else None

    print(f"\n{fecha.date()} ({fila['dia_semana']}): +{fila['subscribers_gained']} suscriptores")
    print(f'  Categorías publicadas ese día: {categorias}')
    if subs_siguiente is not None:
        variacion = subs_siguiente - fila['subscribers_gained']
        print(f'  Día siguiente: +{subs_siguiente} suscriptores ({variacion:+d} respecto al día pico)')
    else:
        print('  Día siguiente fuera del periodo analizado.')

## 6. ¿El calendario de publicación aprovecha el pico de audiencia (sábado)?

En la Fase 8 vimos que el día con más audiencia media es el sábado. Comprobamos si el canal realmente concentra sus publicaciones ahí.

In [ ]:
distribucion_publicacion = video['dia_semana_publicacion'].value_counts(normalize=True).mul(100).round(1)
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
distribucion_publicacion = distribucion_publicacion.reindex(orden_dias)

print('% de vídeos publicados por día de la semana:')
display(distribucion_publicacion)

pct_sabado = distribucion_publicacion.get('Saturday', 0)
esperado_sin_sesgo = 100 / 7
diferencia_pp = esperado_sin_sesgo - pct_sabado

if diferencia_pp > 3:
    print(f"\nDesalineación relevante: solo el {pct_sabado:.1f}% de los vídeos se publica en sábado "
          f"({diferencia_pp:.1f} puntos por debajo de lo esperado sin sesgo, {esperado_sin_sesgo:.1f}%), "
          f"pese a ser el día de mayor audiencia media del canal.")
elif diferencia_pp < -3:
    print(f"\nEl calendario ya sobrerrepresenta el sábado ({pct_sabado:.1f}% de los vídeos, "
          f"{-diferencia_pp:.1f} puntos por encima de lo esperado), coherente con ser el día de mayor audiencia.")
else:
    print(f"\nSin desalineación relevante: el {pct_sabado:.1f}% de los vídeos se publica en sábado, "
          f"cercano a lo esperado sin sesgo ({esperado_sin_sesgo:.1f}%). La diferencia ({diferencia_pp:+.1f} pp) "
          f"es demasiado pequeña para considerarla una desalineación real.")

In [ ]:
plt.figure(figsize=(8, 4))
distribucion_publicacion.plot(kind='bar', color=['#2E6E9E' if d != 'Saturday' else '#B23A2E' for d in orden_dias])
plt.ylabel('% de vídeos publicados')
plt.title('Calendario de publicación por día de la semana (sábado en rojo)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Punto adicional — Tendencia reciente vs. histórico

Añadido porque es la pregunta que más urge al cliente: dentro de los últimos 30 días, ¿el canal está **mejorando o empeorando**? Sin saber la dirección de la tendencia, cualquier recomendación puede llegar tarde o sobrar.

In [ ]:
# Pendiente de la recta de tendencia sobre vistas e ingresos diarios
dias_num = np.arange(len(evolucion))
pendiente_views, _ = np.polyfit(dias_num, evolucion['views'], 1)
pendiente_subs, _ = np.polyfit(dias_num, evolucion['suscriptores_netos'], 1)
pendiente_ingresos, _ = np.polyfit(dias_num, ingresos.sort_values('fecha')['estimated_revenue'], 1)

print(f"Tendencia de vistas diarias: {pendiente_views:+.1f} vistas/día de media a lo largo del periodo")
print(f"Tendencia de suscriptores netos diarios: {pendiente_subs:+.2f} subs/día")
print(f"Tendencia de ingreso diario: {pendiente_ingresos:+.3f} EUR/día")

direccion = 'MEJORANDO' if pendiente_views > 0 and pendiente_ingresos > 0 else ('EMPEORANDO' if pendiente_views < 0 and pendiente_ingresos < 0 else 'MIXTA (sin dirección clara)')
print(f"\nDirección general del periodo: {direccion}")

In [ ]:
# Comparación directa: primera quincena vs segunda quincena del periodo de 30 días
mitad = len(evolucion) // 2
primera_quincena = evolucion.iloc[:mitad]
segunda_quincena = evolucion.iloc[mitad:]

comparativa_quincenas = pd.DataFrame({
    'Primera quincena': [primera_quincena['views'].mean(), primera_quincena['suscriptores_netos'].mean()],
    'Segunda quincena': [segunda_quincena['views'].mean(), segunda_quincena['suscriptores_netos'].mean()],
}, index=['Vistas medias/día', 'Suscriptores netos medios/día'])
display(comparativa_quincenas.round(1))

## Conclusiones y recomendaciones accionables

*(Completar tras revisar los números reales de tu ejecución — se deja la estructura y el criterio de cada recomendación.)*

1. **Días fuertes:** si los 2 días de mayor audiencia caen en fin de semana, coordinar ahí las publicaciones de mayor esfuerzo editorial (no solo mantener el ritmo habitual).
2. **Categoría Afición:** decidir si potenciarla en base a su engagement real (`ratio_likes_vista`), no solo a su volumen de vistas — y solicitar datos de retención/ingreso por categoría en una futura integración real con la Analytics API para confirmar la decisión con más certeza.
3. **Vídeos virales:** si comparten categoría o día de la semana de publicación (sección 3), ese patrón es replicable; si no comparten nada claro, tratarlos como éxitos puntuales, no como una fórmula.
4. **Vídeos "perfectos":** usar el Top 5 de la sección 4 (ya sobre los 338 vídeos, con retención incluida) como referencia de qué combinación de formato/categoría/duración funciona mejor de forma integral; sumar monetización real por vídeo requeriría ampliar la Fase 3 con dimensión `video` en los informes de ingresos.
5. **Ganancia de suscriptores:** si el día siguiente a un pico de suscriptores cae por debajo de la media, valorar publicar contenido de seguimiento (parte 2, respuesta a comentarios) para no perder el impulso.
6. **Calendario vs. audiencia real:** si el sábado está infrarrepresentado en el calendario de publicación pese a ser el día de mayor audiencia, es la recomendación más simple y de mayor impacto potencial de todo este notebook: mover publicaciones clave a esa ventana.
7. **Tendencia:** si la dirección general es 'EMPEORANDO', la urgencia no es solo optimizar contenido, sino frenar la caída antes de escalar cualquier otra recomendación; si es 'MEJORANDO', las recomendaciones anteriores deben enfocarse en acelerar lo que ya funciona, no en corregir una crisis.